In [1]:
import torch
import torch.nn as nn
import torchvision
from torchvision import transforms
from torchvision.datasets import CIFAR10
import numpy as np
from torch.utils.data import random_split, DataLoader

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [12]:
train_val = [0.75, 0.25]
batch_size = 64
l_r = 0.001
epochs = 200
input_channels = 3
no_of_classes = 10

# Data Preprocessing

In [4]:
train_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, ), (0.5,)), transforms.RandomAffine(10, translate=(0.05, 0.05))])
test_transform = transforms.Compose([transforms.Normalize((0.5, ), (0.5,)), transforms.ToTensor()])

train_data = CIFAR10(root="./dataset", train=True, download=True, transform=train_transform)
test_data = CIFAR10(root="./dataset", train=False, download=True, transform=test_transform)

train_data, val_data = random_split(train_data, train_val)

In [5]:
train_dl = DataLoader(train_data, batch_size=batch_size, shuffle=True)
test_dl = DataLoader(test_data, batch_size=batch_size)
val_dl = DataLoader(val_data, batch_size=batch_size)

In [6]:
class ResNetBlock(nn.Module):
    def __init__(self, input_size, output_size, strides=1, conv1x1=False):
        super().__init__()
        self.conv1x1 = conv1x1
        
        self.conv1 = nn.Conv2d(input_size, output_size, kernel_size=3, padding=1, stride=strides)
        self.batchnorm1 = nn.BatchNorm2d(output_size)
        self.relu = nn.ReLU()
        
        self.conv2 = nn.Conv2d(output_size, output_size, kernel_size=3, padding=1)
        self.batchnorm2 = nn.BatchNorm2d(output_size)
        
        if conv1x1:
            self.conv3 = nn.Conv2d(input_size, output_size, kernel_size=1, stride=strides) 

    def forward(self, X): 
        Y = self.conv1(X)
        Y = self.batchnorm1(Y)
        Y = self.relu(Y)
        
        Y = self.conv2(Y)
        Y = self.batchnorm2(Y)
        if self.conv1x1:
            X = self.conv3(X)
            
        Y += X
        return self.relu(Y) 

In [7]:
class ResNetStage(nn.Module):
    def __init__(self, input_size, output_size, no_of_blocks, first_stage=False):
        super().__init__()
        blocks =[]

        if first_stage:
            blocks.append(ResNetBlock(input_size, output_size, strides=1))
        else:
            blocks.append(ResNetBlock(input_size, output_size, strides=2, conv1x1=True))
            
        for _ in range(no_of_blocks-1):
            blocks.append(ResNetBlock(output_size, output_size))

        self.stage = nn.Sequential(*blocks)

    def forward(self, X):
        return self.stage(X)

In [8]:
class RishNet(nn.Module):
    def __init__(self, input_size, output_size):
        super().__init__()

        self.stem = nn.Sequential(nn.Conv2d(input_size, 32, kernel_size=3), nn.BatchNorm2d(32))

        stages = []
        
        stages.append(ResNetStage(32, 32, 3, True))
        stages.append(ResNetStage(32, 64, 3))
        stages.append(ResNetStage(64, 128, 3))

        self.stages = nn.Sequential(*stages)

        self.gap = nn.AdaptiveAvgPool2d((1,1))
        self.flatten = nn.Flatten()

        self.fc = nn.Sequential(nn.Linear(128, 10))

    def forward(self, X):
        X = self.stem(X)
        X = self.stages(X)
        X = self.gap(X)
        X = self.flatten(X)
        Y = self.fc(X)

        return Y
        

In [13]:
model = RishNet(input_channels, no_of_classes).to(device)
optimizer = torch.optim.AdamW(
    model.parameters(), 
    lr=l_r,  
    weight_decay=1e-3
)
criterion = nn.CrossEntropyLoss()

In [ ]:
def acc_of_dl(dl, model, optimizer , loss_fn):
    for X, y in 

In [14]:
def train(dl, epochs, model, optimizer, loss_fn):
    model.train()

    for epoch in range(epochs):
        running_loss = 0
        crt_pred = 0
        preds = 0

        for X, y in dl:
            X = X.to(device)
            y = y.to(device)
            
            y_hat = model(X)
            loss = loss_fn(y_hat, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            out = torch.argmax(y_hat, dim=1)
            crt_pred += (out == y).sum().item()
            preds += y.size(0)

        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {running_loss / len(dl):.4f} | "
            f"Accuracy: {crt_pred / preds:.4f}"
        )

With weight decay of 1e-2 --> 94.08 % 
With lr of 0.005 -----------> 93.55 % couldn't properly converge


In [15]:
train(train_dl, epochs, model, optimizer, criterion)

Epoch 01 | Loss: 1.4042 | Accuracy: 0.4843
Epoch 02 | Loss: 1.0125 | Accuracy: 0.6397
Epoch 03 | Loss: 0.8271 | Accuracy: 0.7085
Epoch 04 | Loss: 0.7205 | Accuracy: 0.7474
Epoch 05 | Loss: 0.6387 | Accuracy: 0.7761
Epoch 06 | Loss: 0.5807 | Accuracy: 0.7970
Epoch 07 | Loss: 0.5283 | Accuracy: 0.8169
Epoch 08 | Loss: 0.4853 | Accuracy: 0.8309
Epoch 09 | Loss: 0.4459 | Accuracy: 0.8448
Epoch 10 | Loss: 0.4236 | Accuracy: 0.8512
Epoch 11 | Loss: 0.3883 | Accuracy: 0.8635
Epoch 12 | Loss: 0.3669 | Accuracy: 0.8711
Epoch 13 | Loss: 0.3420 | Accuracy: 0.8815
Epoch 14 | Loss: 0.3169 | Accuracy: 0.8871
Epoch 15 | Loss: 0.2996 | Accuracy: 0.8945
Epoch 16 | Loss: 0.2792 | Accuracy: 0.9023
Epoch 17 | Loss: 0.2627 | Accuracy: 0.9064
Epoch 18 | Loss: 0.2493 | Accuracy: 0.9107
Epoch 19 | Loss: 0.2329 | Accuracy: 0.9171
Epoch 20 | Loss: 0.2216 | Accuracy: 0.9209
Epoch 21 | Loss: 0.2140 | Accuracy: 0.9238
Epoch 22 | Loss: 0.1976 | Accuracy: 0.9290
Epoch 23 | Loss: 0.1874 | Accuracy: 0.9340
Epoch 24 | 